In [11]:
from ultralytics import YOLO
import cv2
import os
from threading import Thread

In [12]:
output_folder = "saved_frames"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [13]:
# модели ниже версией плохо справляються с поставленой задачей, остальные версии 8й слишком тяжелые и не справляються с обработкой в реальном времени или за тот же промежуток времени
model = YOLO("yolov8s.pt")

In [14]:
cap=cv2.VideoCapture("video_2025-01-19_21-33-09.mp4")



roi=[(130, 130), (600, 600)]  #областъ интереса

frame_count=0
batch_size=8  # при батче больше 8 обработка замедляется
frames=[]
Compressionione=[]
skip_counter=0

In [15]:
def process_batch(frames, Compressionione):
    global frame_count

    # детекция в батче
    results = model(frames)

    for i in range(len(frames)):
        frame = frames[i]
        height, width = Compressionione[i]

        # обработка каждого обнаруженного объекта
        for box in results[i].boxes:
            class_name = results[i].names[int(box.cls)] 
            if class_name == "car":
                x1, y1, x2, y2 = map(int, box.xyxy[0]) 

                # box с ROI 
                if (x1 < roi[1][0] and x2 > roi[0][0] and y1 < roi[1][1] and y2 > roi[0][1]) or \
                   (x1 >= roi[0][0] and y1 >= roi[0][1] and x2 <= roi[1][0] and y2 <= roi[1][1]):
                    frame_count += 1 

                    
                    roi_frame = frame[roi[0][1]:roi[1][1], roi[0][0]:roi[1][0]]

                    # сохраняем кадр
                    output_path = os.path.join(output_folder, f"frame_{frame_count}.jpg")
                    cv2.imwrite(output_path, roi_frame)
                    print(f"Сохранен кадр: {output_path}")

        # Рисуем ROI и box на кадре
        cv2.rectangle(frame, roi[0], roi[1], (0, 255, 0), 2)  
        for box in results[i].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2) 

        # Показываем кадр
        cv2.imshow("Frame", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break



In [16]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Пропуск каждого второго кадра (для обработки в реальном времени без отставания)
    skip_counter += 1
    if skip_counter % 2 != 0:
        continue

    
    height, width = frame.shape[:2]
    new_width = int(width * 0.8)
    new_height = int(height * 0.8)
    frame = cv2.resize(frame, (new_width, new_height))# уменьшение кадра на 20%

   
    frames.append(frame) # Добавление кадра в батч
    Compressionione.append((new_height, new_width))

    # Если батч заполнен, обрабатываем его в отдельном потоке
    if len(frames) == batch_size:
        thread = Thread(target=process_batch, args=(frames.copy(), Compressionione.copy()))# Создание потока для обработки батча
        thread.start()

        # Очистка батча
        frames = []
        Compressionione = []

cap.release()
cv2.destroyAllWindows()


Ultralytics 8.3.64  Python-3.12.7 torch-2.5.1+cpu CPU (13th Gen Intel Core(TM) i7-13700H)

Ultralytics 8.3.64  Python-3.12.7 torch-2.5.1+cpu CPU (13th Gen Intel Core(TM) i7-13700H)

Ultralytics 8.3.64  Python-3.12.7 torch-2.5.1+cpu CPU (13th Gen Intel Core(TM) i7-13700H)

Ultralytics 8.3.64  Python-3.12.7 torch-2.5.1+cpu CPU (13th Gen Intel Core(TM) i7-13700H)




YOLOv8s summary (fused): 168 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs


YOLOv8s summary (fused): 168 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs

YOLOv8s summary (fused): 168 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
YOLOv8s summary (fused): 168 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs


0: 640x384 1 suitcase, 118.4ms
1: 640x384 (no detections), 118.4ms
2: 640x384 (no detections), 118.4ms
3: 640x384 (no detections), 118.4ms
4: 640x384 (no detections), 118.4ms
5: 640x384 1 suitcase, 118.4ms
6: 640x384 (no detections), 118.4ms
7: 640x384 (no detections), 118.4ms
Speed: 4.

0: 640x384 1 traffic light, 122.7ms
1: 640x384 1 traffic light, 122.7ms
2: 640x384 (no detections), 122.7ms
3: 640x384 (no detections), 122.7ms
4: 640x384 1 traffic light, 122.7ms
5: 640x384 1 traffic light, 122.7ms
6: 640x384 (no detections), 122.7ms
7: 640x384 (no detections), 122.7ms
Speed: 26.6ms preprocess, 122.7ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)
0: 640x384 (no detections), 82.7ms
1: 640x384 (no detections), 82.7ms
2: 640x384 (no detections), 82.7ms
3: 640x384 (no detections), 82.7ms
4: 640x384 (no detections), 82.7ms
5: 640x384 (no detections), 82.7ms
6: 640x384 (no detections), 82.7ms
7: 640x384 (no detections), 82.7ms
Speed: 7.6ms preprocess, 82.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)
0: 640x384 1 clock, 66.6ms
1: 640x384 (no detections), 66.6ms
2: 640x384 (no detections), 66.6ms
3: 640x384 (no detections), 66.6ms
4: 640x384 1 clock, 66.6ms
5: 640x384 (no detections), 66.6ms
6: 640x384 (no detections), 66.6ms
7: 640x